In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
from glob import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool

In [4]:
from slidingwinalignment.io import read_fasta
from slidingwinalignment.matrices import similarity_matrix
from slidingwinalignment.alignment import transform_similarity_matrix, fitting_alignment

## configuration

Please adjust the number of CPU workers based on your machine.

Set `N_CPU = 1` to run the screening step without multiprocessing.

In [5]:
# User-configurable parameters
N_CPU = 4

## read sequences from fasta

In [6]:
path = "data/SH3.fasta"
all_seqs = read_fasta(path)

Read 10 sequences.
Example:
2J6K_0
MVDYIVEYDYDAVHDDELTIRVGEIIRNVKKLQEEGWLEGELNGRRGMFPDNFVKEIKRETEFKDDSLPIKRERHGNVASLVQRISTYGLPAGGIQPHPQTKNIKKKTKKRQCKVLFEYIPQNEDELELKVGDIIDINEEVEEGWWSGTLNNKLGLFPSNFVKELEVTDDGETHEAQDDSETVLAGPTSPIPSLGNVSETASGSVTQPKKIRGIGFGDIFKEGSVKLRTRTSSSETEEKKPEKPLILQSLGPKTQSVEITKTDTEGKIKAKEYCRTLFAYEGTNEDELTFKEGEIIHLISKETGEAGWWRGELNGKEGVFPDNFAVQINELDKDFPKPKKPPPPAKAPAPKPELIAAEKKYFSLKPEEKDEKSTLEQKPSKPAAPQVPPKKPTPPTKASNLLRSSGTVYPKRPEKPVPPPPPIAKINGEVSSISSKFETEPVSKLKLDSEQLPLRPKSVDFDSLTVRTSKETDVVNFDDIASSENLLHLTANRPKMPGRRLPGRFNGGHSPTHSPEKILKLPKEEDSANLKPSELKKDTCYSPKPSVYLSTPSSASKANTTAFLTPLEIKAKVETDDVKKNSLDELRAQIIELLCIVEALKKDHGKELEKLRKDLEEEKTMRSNLEMEIEKLKKAVLSS


## load model and get embedding

In [7]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import re

In [8]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
tokenizer = T5Tokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/ProstT5").to(device)
model = model.float() if device.type=='cpu' else model.half()
print("Model loaded.")

Using device: cuda:0


Loading weights: 100%|██████████| 195/195 [00:00<00:00, 56480.17it/s]


Model loaded.


In [9]:
# key and value to two separate lists
keys = list(all_seqs.keys())
values = list(all_seqs.values())

In [10]:
seq_embeddings = []
for i in tqdm(values):
    sequences = [i]
    length = len(sequences[0])
    sequences = [" ".join(list(re.sub(r"[UZOB]", "X", sequence))) for sequence in sequences]
    sequences = [ "<AA2fold>" + " " + s if s.isupper() else "<fold2AA>" + " " + s # this expects 3Di sequences to be already lower-case
                      for s in sequences
                    ]
    ids = tokenizer(sequences,
                    add_special_tokens=True,
                    padding="longest",
                    return_tensors='pt').to(device)
    with torch.no_grad():
      embedding_repr = model(
              ids.input_ids, 
              attention_mask=ids.attention_mask
              )
    emb = embedding_repr.last_hidden_state[0, 1 : length + 1].cpu().numpy()
    seq_embeddings.append(emb)

100%|██████████| 10/10 [00:02<00:00,  4.63it/s]


In [11]:
len(seq_embeddings), len(seq_embeddings[0]), seq_embeddings[0].shape

(10, 639, (639, 1024))

In [12]:
all_embs = dict(zip(keys, seq_embeddings))

# use 1KIK as the target pdb in this example

In [13]:
searching_motif_emb = all_embs['1KIK_6']

In [14]:
window_size = 3

In [15]:
_searching = None
_window = None

def _init(searching_motif_emb, window_size):
    global _searching, _window
    _searching = searching_motif_emb
    _window = window_size

def _worker(item):
    key, emb = item
    try:
        sim_matrix = similarity_matrix(_searching, emb, _window)
        return key, sim_matrix
    except Exception as e:
        print(f"Error processing key {key}: {e}")
        return key, None

items = list(all_embs.items())

with Pool(processes=N_CPU, initializer=_init, initargs=(searching_motif_emb, window_size)) as pool:
    it = pool.imap_unordered(_worker, items, chunksize=1)
    matrix_data = dict(tqdm(it, total=len(items)))

100%|██████████| 10/10 [00:00<00:00, 15.40it/s]


## define the target motif region

Set `start` and `end` to the residue indices corresponding to the target motif region.

In [18]:
start = 61
end = 121

## screening

In [19]:
_matrix = None
_window = None

def _init(matrix_data, window_size):
    global _matrix, _window
    _matrix = matrix_data
    _window = window_size

def _worker(matrix_name):
    matrix = _matrix[matrix_name]
    if matrix is None:
        return matrix_name, None, None
    window_size = _window

    matrix = transform_similarity_matrix(matrix, midpoint=0.2, sharpness=9, scale=3)
    range1 = [start, end - window_size + 1]
    matrix = matrix[range1[0]:range1[1], :]

    score, protein1, protein2 = fitting_alignment(matrix, 10)
    return matrix_name, score, (int(protein2[0]), int(protein2[1]) + window_size)

items = list(matrix_data.keys())

with Pool(processes=N_CPU, initializer=_init, initargs=(matrix_data, window_size)) as pool:
    it = pool.imap_unordered(_worker, items, chunksize=1)
    rows = list(tqdm(it, total=len(items)))

results_df = pd.DataFrame(rows, columns=["name", "score", "range"])

100%|██████████| 10/10 [00:00<00:00, 12.81it/s]


In [20]:
results_df = results_df.sort_values("score", ascending=False)
results_df

,name,score,range
4,1KIK_6,173.740382,"(61, 121)"
5,1ABQ_5,161.884456,"(61, 121)"
2,5QU3_2,154.251571,"(2, 62)"
3,2J6K_0,150.720006,"(108, 168)"
9,1E6G_1,144.273797,"(967, 1027)"
6,1X2K_9,138.332116,"(12, 72)"
7,1ZUK_8,126.118270,"(6, 68)"
1,1M30_4,124.935308,"(132, 192)"
8,5XGG_7,113.248001,"(992, 1049)"
0,1H3H_3,109.886862,"(0, 56)"
